# ⚽ FIFA Women's World Cup Analysis
## Goal Distribution in the FIFA Women’s World Cup (2019 vs 2023)

This analysis explores how goals are distributed among teams and whether offensive production is concentrated in a few dominant teams.

## 🎯 Objective

- Identify top scoring teams in each edition  
- Compare changes between 2019 and 2023  
- Measure goal concentration among top teams  

In [ ]:
# ============================================
# 📦 Imports
# ============================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ============================================
# 📂 Load Data
# ============================================

df_2019 = pd.read_json("dataset_2019.json")
df_2023 = pd.read_json("dataset_2023.json")

df_2019["year"] = 2019
df_2023["year"] = 2023

df = pd.concat([df_2019, df_2023], ignore_index=True)

In [ ]:
# Transform nested JSON into team-level goal data
# extract team names
df["home_team"] = df["home_team"].apply(lambda x: x["home_team_name"])
df["away_team"] = df["away_team"].apply(lambda x: x["away_team_name"])

# long format (team + goals)
home = df[["year", "home_team", "home_score"]].copy()
home.columns = ["year", "team", "goals"]

away = df[["year", "away_team", "away_score"]].copy()
away.columns = ["year", "team", "goals"]

df_long = pd.concat([home, away], ignore_index=True)

# clear names
df_long["team"] = df_long["team"].str.replace(" Women's", "", regex=False)

In [ ]:
# Aggregate goals scored by each team
goals_by_team = (
    df_long.groupby(["year", "team"])["goals"]
    .sum()
    .reset_index()
)

goals_by_team.head()

## ⚽ Top Scoring Teams

First, we identify the teams with the highest number of goals in each edition.

In [ ]:
# ============================================
# 📊 Top 10 Teams by Goals
# ============================================

top10_2019 = (
    goals_by_team[goals_by_team["year"] == 2019]
    .sort_values(by="goals", ascending=False)
    .head(10)
)

top10_2023 = (
    goals_by_team[goals_by_team["year"] == 2023]
    .sort_values(by="goals", ascending=False)
    .head(10)
)

top10_2019, top10_2023

In 2019, the United States led by a large margin, indicating strong concentration of goals among top teams.

In 2023, Spain leads the ranking, but with a smaller gap to other teams, suggesting a more balanced offensive distribution.

## 📊 Goal Concentration

We measure how much of the total goals are scored by the top teams.

In [24]:
def top_n_share(df, year, n=3):
    data = df[df["year"] == year]
    total = data["goals"].sum()

    top_n = (
        data.sort_values(by="goals", ascending=False)
        .head(n)["goals"]
        .sum()
    )

    return top_n / total

In [ ]:
share_2019 = top_n_share(goals_by_team, 2019, 3)
share_2023 = top_n_share(goals_by_team, 2023, 3)

print("Top 3 - 2019:", round(share_2019 * 100, 1), "%")
print("Top 3 - 2023:", round(share_2023 * 100, 1), "%")

The top 3 teams scored:

- 34.9% of total goals in 2019  
- 28.7% in 2023  

👉 This indicates a decrease in goal concentration.


## 🔗 Changes Between Editions (Dumbbell Chart)

This chart shows how goal scoring changed for top teams between 2019 and 2023.

In [ ]:
# ============================================
# 🔗 Dumbbell Chart
# Compare goal production between editions
# ============================================

df_2019 = goals_by_team[goals_by_team["year"] == 2019]
df_2023 = goals_by_team[goals_by_team["year"] == 2023]

comparison = pd.merge(
    df_2019,
    df_2023,
    on="team",
    how="outer",
    suffixes=("_2019", "_2023")
).fillna(0)

# Select Top 10 teams based on 2023 goals
top10 = comparison.sort_values(by="goals_2023", ascending=False).head(10)

# Sort for better visualization
top10 = top10.sort_values(by="goals_2023")

y = np.arange(len(top10))

plt.figure(figsize=(8,6))

# Connecting lines
for i in range(len(top10)):
    plt.plot(
        [top10["goals_2019"].iloc[i], top10["goals_2023"].iloc[i]],
        [y[i], y[i]],
        color="#adb5bd",
        linewidth=2
    )

# Points
plt.scatter(top10["goals_2019"], y, color="#a8dadc", label="2019", s=80)
plt.scatter(top10["goals_2023"], y, color="#d62828", label="2023", s=80)

plt.yticks(y, top10["team"])
plt.xlabel("Goals")
plt.title("Top 10 Teams — Goals Comparison (2019 vs 2023)")
plt.legend()
plt.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## 📈 Goal Concentration (Pareto Chart)

This chart shows how goals are distributed across teams and how quickly they accumulate.

In [ ]:
# ============================================
# 📈 Pareto Analysis
# Goal concentration comparison
# ============================================

plt.figure(figsize=(8,6))

def pareto_full(df, year):
    data = (
        df[df["year"] == year]
        .sort_values(by="goals", ascending=False)
        .reset_index(drop=True)
    )
    data["cum_pct"] = data["goals"].cumsum() / data["goals"].sum()
    data["rank"] = range(1, len(data) + 1)
    return data

p2019 = pareto_full(goals_by_team, 2019)
p2023 = pareto_full(goals_by_team, 2023)

plt.plot(p2019["rank"], p2019["cum_pct"], label="2019", linewidth=2)
plt.plot(p2023["rank"], p2023["cum_pct"], label="2023", linewidth=2, linestyle="--")

plt.axhline(0.5, linestyle='--', alpha=0.5)

plt.xlabel("Número de seleções")
plt.ylabel("% acumulado de gols")
plt.title("Concentração de gols — 2019 vs 2023")

plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## 📦 Goal Distribution Across Teams

This chart compares how goals are distributed among teams in each edition.

In [ ]:
# ============================================
# 📦 Boxplot
# Compare goal distribution between editions
# ============================================

plt.figure(figsize=(6,6))

box = plt.boxplot(
    [
        goals_by_team[goals_by_team["year"] == 2019]["goals"],
        goals_by_team[goals_by_team["year"] == 2023]["goals"]
    ],
    labels=["2019", "2023"],
    patch_artist=True,
    widths=0.5,
    flierprops=dict(
        marker='o',
        markersize=8,
        markerfacecolor='#d62828',
        alpha=0.7
    )
)

# Apply consistent colors
colors = ["#a8dadc", "#d62828"]

for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

# Style lines
for element in ['whiskers', 'caps', 'medians']:
    for item in box[element]:
        item.set_color("#6c757d")

plt.title("Goal Distribution Across Teams (2019 vs 2023)")
plt.ylabel("Goals")

plt.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

# high quality image
plt.savefig("nome_do_grafico.png", dpi=300, bbox_inches='tight')

## 🧠 Final Insight

The 2023 Women's World Cup showed a less concentrated offensive landscape compared to 2019.

While traditional powerhouses remained important, a larger number of teams contributed to total goals scored.

👉 This suggests a more balanced and competitive tournament structure.